# Rename Engine Playground

Interactive exploration of every rename operation in the engine.
Each section is self-contained — tweak the config, run the cell, see the preview.

**Nothing is renamed until you explicitly uncomment and run an execute cell.**

### Operations covered
| # | Operation | What it does |
|---|---|---|
| 1 | `replace_text` | Find and replace text in filenames |
| 2 | `regex_replace` | Regex substitution with backreferences |
| 3 | `strip_text` | Remove prefix, suffix, or any substring |
| 4 | `change_case` | Upper / lower / title / sentence case |
| 5 | `change_extension` | Modify file extensions |
| 6 | `remove_chars` | Remove characters by position or delimiter |
| 7 | `insert_at` | Insert text at a position in the stem |
| 8 | `add_prefix` / `add_suffix` | Prepend or append text |
| 9 | `auto_date` | Add file date to filenames |
| 10 | `append_folder_name` | Add parent folder name |
| 11 | `sequential_name` | Sequential numbering |
| 12 | `change_name` | Fixed text, reverse, or remove stem |
| 13 | `word_space` | Split camelCase / underscores / hyphens |

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
src_dirs = [
    "packages/core/src",
    "packages/fs-indexer/src",
    "packages/metadata/src",
    "packages/media-grouping/src",
    "packages/duplicate-detection/src",
    "packages/rename-engine/src",
    "packages/job-runner/src",
    "packages/ui-common/src",
    "apps/media-curator/src",
    "apps/disk-atlas/src",
    "apps/rename-studio/src",
    "apps/control-center/src",
]
for d in src_dirs:
    p = str(ROOT / d)
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"Project root: {ROOT}")

## Setup: Pick a Directory

Point this at any folder you want to experiment with.
All operations below will use these entries.

In [ ]:
from tidyforge.fs_indexer import ExtensionFilter, scan_directory

# ---- EDIT THIS ----
SOURCE_DIR = Path(".")  # change to your target folder
EXT_FILTER = None        # e.g. {".jpg", ".png"} or None for all
# -------------------

source = Path(SOURCE_DIR).resolve()
filters = []
if EXT_FILTER:
    filters.append(ExtensionFilter(include=EXT_FILTER))

entries = list(scan_directory(source, filters=filters, max_depth=0))
print(f"Found {len(entries):,} files in {source}\n")
for e in entries[:15]:
    print(f"  {e.name}")
if len(entries) > 15:
    print(f"  ... and {len(entries) - 15:,} more")

### Helper: preview any plan

Run this cell once — then use `show(plan)` anywhere below.

In [ ]:
def show(plan, limit=20):
    """Pretty-print a RenamePlan."""
    issues = plan.validate()
    if issues:
        print(f"\u26a0 {len(issues):,} issues:")
        for i in issues[:5]:
            print(f"  {i}")
        if len(issues) > 5:
            print(f"  ... and {len(issues) - 5:,} more")
        print()
    if not plan.actions:
        print("No files to rename.")
        return
    print(f"{len(plan.actions):,} renames planned:\n")
    for a in plan.actions[:limit]:
        print(f"  {a.source.name}")
        print(f"    -> {a.destination.name}")
    if len(plan.actions) > limit:
        print(f"  ... and {len(plan.actions) - limit:,} more")

print("show() helper ready.")

---

## 1. Replace Text

Find and replace text in filenames. Supports case-insensitive
and first-occurrence-only modes.

In [ ]:
from tidyforge.rename_engine import replace_text

plan = replace_text(
    entries,
    old="IMG",
    new="PHOTO",
    # case_sensitive=True,
    # first_only=False,
)
show(plan)

## 2. Regex Replace

Regex substitution with backreference support.
Use `include_ext=False` to apply regex only to the stem.

In [ ]:
from tidyforge.rename_engine import regex_replace

plan = regex_replace(
    entries,
    pattern=r"IMG_(\d{8})_(\d+)",
    replacement=r"\1-\2",
    # include_ext=True,
)
show(plan)

## 3. Strip Text

Remove a prefix, suffix, or any occurrence of a substring.

In [ ]:
from tidyforge.rename_engine import strip_text

plan = strip_text(
    entries,
    text="Screenshot_",
    position="prefix",   # "prefix", "suffix", or "any"
    # case_sensitive=True,
)
show(plan)

## 4. Change Case

Transform the case of the stem, extension, or both.
Modes: `upper`, `lower`, `title`, `sentence`.

In [ ]:
from tidyforge.rename_engine import change_case

plan = change_case(
    entries,
    case="lower",     # "upper", "lower", "title", "sentence"
    scope="stem",     # "stem", "ext", "both"
)
show(plan)

## 5. Change Extension

Modify file extensions: change case, replace, append, or remove.

In [ ]:
from tidyforge.rename_engine import change_extension

plan = change_extension(
    entries,
    mode="lower",     # "lower", "upper", "title", "fixed", "extra", "remove"
    # new_ext=".bak",  # for "fixed" or "extra" modes
)
show(plan)

## 6. Remove Characters

Remove characters from the filename stem by count,
position range, or cropping around a delimiter.

In [ ]:
from tidyforge.rename_engine import remove_chars

# Remove first 11 characters (e.g. "Screenshot_")
plan = remove_chars(entries, first_n=11)
show(plan)

# Other examples (uncomment one at a time):
# plan = remove_chars(entries, last_n=5)
# plan = remove_chars(entries, from_pos=2, to_pos=6)
# plan = remove_chars(entries, crop_before="_")
# plan = remove_chars(entries, crop_after="_")

## 7. Insert Text at Position

Insert text at a specific character position in the stem.
Negative positions count from the end.

In [ ]:
from tidyforge.rename_engine import insert_at

plan = insert_at(
    entries,
    text="2024_",
    position=0,    # 0 = start, -1 = before last char
)
show(plan)

## 8. Add Prefix / Add Suffix

Simple prepend or append operations.

In [ ]:
from tidyforge.rename_engine import add_prefix, add_suffix  # noqa: F401

# Prefix
plan = add_prefix(entries, "backup_")
show(plan)

# Suffix (added before the extension)
# plan = add_suffix(entries, "_v2")
# show(plan)

## 9. Auto Date

Add the file's modification or creation date to the filename.

In [ ]:
from tidyforge.rename_engine import auto_date

plan = auto_date(
    entries,
    date_source="modified",   # "modified" or "created"
    fmt="%Y-%m-%d",           # any strftime format
    position="prefix",        # "prefix" or "suffix"
    separator="_",
)
show(plan)

## 10. Append Folder Name

Prepend or append the parent folder name to filenames.
Use `levels` to include grandparent, etc.

In [ ]:
from tidyforge.rename_engine import append_folder_name

plan = append_folder_name(
    entries,
    position="prefix",   # "prefix" or "suffix"
    separator="_",
    levels=1,            # 1 = parent, 2 = grandparent + parent
)
show(plan)

## 11. Sequential Numbering

Rename files with sequential numbers. Supports different bases
(hex, octal, binary), prefix/suffix positioning, and per-folder reset.

In [ ]:
from tidyforge.rename_engine import sequential_name

# Replace entire name with counter
plan = sequential_name(entries)
show(plan)

# As prefix, keeping original name
# plan = sequential_name(
#     entries,
#     position="prefix",
#     separator="_",
#     start=1,
#     increment=1,
#     pad=3,
#     base=10,            # 10, 16 (hex), 8, 2 (binary)
#     reset_per_folder=False,
# )
# show(plan)

## 12. Change Name

Replace, reverse, or remove the entire filename stem.

In [ ]:
from tidyforge.rename_engine import change_name

# Set all stems to a fixed name
plan = change_name(entries, mode="fixed", fixed_name="image")
show(plan)

# Reverse the stem
# plan = change_name(entries, mode="reverse")
# show(plan)

## 13. Word Space

Split camelCase, underscores, and hyphens into separate words.

In [ ]:
from tidyforge.rename_engine import word_space

plan = word_space(
    entries,
    separator=" ",   # or "-", "_", etc.
)
show(plan)

---

## Execute Any Plan

Once you're happy with a plan from any section above, assign it to
`final_plan` and run the cell below.

**The execute line is commented out for safety.**

In [ ]:
# Assign your chosen plan here:
final_plan = plan  # <- replace 'plan' with the variable from above

# Dry-run first
dry = final_plan.execute(dry_run=True)
print(dry.summary)

# Uncomment to execute for real:
# final_plan = ...  # rebuild fresh (dry-run consumes statuses)
# manifest = final_plan.execute(dry_run=False)
# print(manifest.summary)